# Modelo M03: Riesgo de morosidad (Clasificación multiclase)

**Curso:** Pruebas de Software (CI0142)
**Problema:** estimar el nivel de riesgo (bajo, medio, alto) de una solicitud de préstamo.
**Métricas evaluadas:** accuracy, precision, recall, F1 y matriz de confusión del clasificador binario; proporción de rechazos reales por nivel de riesgo.

Este cuaderno explica paso a paso el entrenamiento. El modelo que consume la API lo genera el script `entrenamiento.py` de esta misma carpeta; el cuaderno no exporta nada, para que exista un solo origen del artefacto. Si el cuaderno y el script llegan a diferir, manda el script.

## 1. El problema con la variable objetivo

La especificación define M03 como clasificación multiclase. La hoja de datos entregada, en cambio, tiene una sola variable objetivo, `Loan_Status`, con dos valores (Y/N), y por las columnas que la acompañan registra si la solicitud fue aprobada, no si el solicitante cayó en morosidad. La hoja `Mor Riesgo Test` no trae esa columna.

Se evaluaron dos formas de obtener tres clases:

- **Alternativa A: etiqueta de tres clases construida con una regla.** Combinar `Loan_Status` con `Credit_History` (por ejemplo, alto = rechazado sin historial). Permite calcular una matriz de confusión de 3×3, pero obliga a sacar `Credit_History` de las entradas (si no, el modelo aprende a repetir la regla) y evalúa al modelo contra una etiqueta inventada por el equipo.
- **Alternativa B (la elegida): clasificador binario y niveles por probabilidad.** El modelo aprende lo observado (aprobado o rechazado) y su probabilidad de rechazo se corta en tres niveles. Todas las columnas pueden ser entradas y no se inventan etiquetas. A cambio, no existe matriz de confusión de 3×3 contra datos reales: los niveles se evalúan con la proporción de rechazos reales en cada uno.

Se eligió B porque enriquece más el proceso de pruebas: la conversión de probabilidad a nivel es una función con oráculo exacto (valores límite en los cortes), y el modelo admite oráculos verificables de dirección, consistencia y calidad. Supuesto que debe declararse: el rechazo se usa como sustituto del riesgo de morosidad.

## 2. Librerías y configuración

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, make_scorer,
    precision_score, recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RUTA_DATOS = Path("../../data/raw/M01-M02-M03.xlsx")
HOJA_ENTRENAMIENTO = "Mor Riesgo Ent"

SEMILLA = 42
PROPORCION_PRUEBA = 0.2
PLIEGUES_CV = 5

CORTE_MEDIO = 0.33
CORTE_ALTO = 0.67
NIVELES = ("bajo", "medio", "alto")

Los cortes de los niveles se fijan aquí, antes de entrenar. Son una decisión del equipo (el intervalo de probabilidad partido en tres tramos iguales), no un resultado de los datos. Fijarlos antes permite diseñar los casos de valores límite sin depender de cada reentrenamiento.

## 3. Carga y exploración

In [2]:
datos = pd.read_excel(RUTA_DATOS, sheet_name=HOJA_ENTRENAMIENTO)
print(f"Registros: {datos.shape[0]}  Columnas: {datos.shape[1]}")
datos.head()

Registros: 614  Columnas: 13


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [3]:
datos.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), object(1), str(7)
memory usage: 62.5+ KB


Hay faltantes en varias columnas y `Dependents` aparece como tipo objeto, señal de que mezcla números con texto. Se revisan ambos puntos.

In [4]:
faltantes = datos.isna().sum()
faltantes[faltantes > 0]

Gender              13
Married              3
Dependents          15
Self_Employed       32
LoanAmount          22
Loan_Amount_Term    14
Credit_History      50
dtype: int64

In [5]:
print(datos["Dependents"].value_counts(dropna=False))
print()
print(datos["Loan_Status"].value_counts(dropna=False))

Dependents
0      345
1      102
2      101
3+      51
NaN     15
Name: count, dtype: int64

Loan_Status
Y    422
N    192
Name: count, dtype: int64


`Dependents` usa el texto `"3+"` para tres o más dependientes; se convertirá a 3. `Loan_Status` solo tiene Y y N, sin vacíos: es la única etiqueta disponible.

La tabla siguiente cruza `Credit_History` con el resultado. Sirve para dos cosas: muestra que el historial crediticio es la columna que más separa aprobados de rechazados, y justifica cómo se imputan sus vacíos (paso 5).

In [6]:
pd.crosstab(datos["Credit_History"].fillna(-1), datos["Loan_Status"], margins=True)

Loan_Status,N,Y,All
Credit_History,,,
-1.0,13,37,50
0.0,82,7,89
1.0,97,378,475
All,192,422,614


La fila `-1` son los vacíos. Su proporción de rechazos se parece a la del grupo con historial 1 y no a la del grupo con historial 0, así que imputarlos con el valor más frecuente (1) es coherente con los datos.

## 4. Variable objetivo y selección de columnas

Se construye `rechazo` = 1 si `Loan_Status` es N. Las entradas son todas las columnas de la solicitud excepto dos:

- `Loan_ID`: es un identificador; no describe al solicitante y solo le permitiría al modelo memorizar filas.
- `Loan_Status`: es la variable objetivo.

No se aplicó una selección estadística de variables (correlaciones o importancia). Ninguna columna se deriva de la etiqueta, así que no hay fuga de información que obligue a excluir otra. `Credit_History` se mantiene como entrada; en la alternativa A habría tenido que salir.

In [7]:
inesperados = set(datos["Loan_Status"].dropna()) - {"Y", "N"}
if inesperados or datos["Loan_Status"].isna().any():
    raise ValueError(f"Loan_Status con valores no esperados: {inesperados}")

datos["Dependents"] = pd.to_numeric(datos["Dependents"].replace("3+", 3))
rechazo = (datos["Loan_Status"] == "N").astype(int).rename("rechazo")
entradas = datos.drop(columns=["Loan_ID", "Loan_Status"])

print(f"Entradas: {list(entradas.columns)}")
print(f"Rechazos: {rechazo.sum()}  Aprobados: {(rechazo == 0).sum()}")

Entradas: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']
Rechazos: 192  Aprobados: 422


## 5. Preprocesamiento dentro del pipeline

Las entradas se separan en dos grupos, siguiendo la preparación de datos del laboratorio (faltantes, tipos y codificación):

- **Numéricas:** se imputan con la mediana (menos sensible a extremos que la media) y se escalan.
- **Categóricas:** se imputan con el valor más frecuente y se codifican con one-hot, porque no tienen un orden que respalde asignarles números consecutivos. `Credit_History` va aquí aunque sea 0/1: representa tener o no historial, no una cantidad.

Todo queda dentro de un `ColumnTransformer` que forma parte del pipeline exportado. Así la API aplica exactamente la misma transformación que el entrenamiento, y la mediana, la moda y la escala se calculan solo con datos de entrenamiento.

`handle_unknown="ignore"` evita que una categoría no vista detenga la predicción: sus columnas quedan en cero. Por eso la validación del contrato (rechazar categorías desconocidas con 400) le corresponde a la API.

In [8]:
COLUMNAS_NUMERICAS = [
    "Dependents", "ApplicantIncome", "CoapplicantIncome",
    "LoanAmount", "Loan_Amount_Term",
]
COLUMNAS_CATEGORICAS = [
    "Gender", "Married", "Education", "Self_Employed",
    "Property_Area", "Credit_History",
]

asignadas = set(COLUMNAS_NUMERICAS) | set(COLUMNAS_CATEGORICAS)
assert asignadas == set(entradas.columns), "Hay columnas sin grupo o inexistentes"


def crear_preprocesador():
    """Imputacion, escalado y one-hot que viajan dentro del modelo exportado."""
    numericas = Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ])
    categoricas = Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numericas, COLUMNAS_NUMERICAS),
        ("cat", categoricas, COLUMNAS_CATEGORICAS),
    ])

La verificación con `assert` importa porque `ColumnTransformer` descarta en silencio cualquier columna que no esté en ninguna lista.

## 6. Partición entrenamiento/prueba

Se reserva el 20 % para la medición final. La partición es estratificada por `rechazo`, para que ambos conjuntos tengan casi la misma proporción de rechazos, y usa una semilla fija para que sea repetible.

In [9]:
x_ent, x_pru, y_ent, y_pru = train_test_split(
    entradas, rechazo,
    test_size=PROPORCION_PRUEBA, stratify=rechazo, random_state=SEMILLA,
)
print(f"Entrenamiento: {len(x_ent)}  Prueba: {len(x_pru)}")
print(f"Tasa de rechazo  total: {rechazo.mean():.3f}  "
      f"entrenamiento: {y_ent.mean():.3f}  prueba: {y_pru.mean():.3f}")

Entrenamiento: 491  Prueba: 123
Tasa de rechazo  total: 0.313  entrenamiento: 0.314  prueba: 0.309


## 7. Candidatos y selección por validación cruzada

Se comparan una línea base (`DummyClassifier`, que siempre predice la clase mayoritaria) y tres clasificadores. Los hiperparámetros son fijos, sin búsqueda; otros valores podrían cambiar el ganador.

El modelo se escoge por el F1 promedio de la validación cruzada de 5 pliegues, calculado **solo con el conjunto de entrenamiento**. El preprocesador se reajusta dentro de cada pliegue. El conjunto de prueba no participa en la elección.

Se usa F1 porque las clases están desbalanceadas: la línea base obtiene una accuracy aceptable sin detectar un solo rechazo.

In [10]:
def armar_pipeline(modelo):
    """Preprocesador y modelo en un solo objeto, que es lo que se exporta."""
    return Pipeline([("preprocesar", crear_preprocesador()), ("modelo", modelo)])


candidatos = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Regresion logistica": LogisticRegression(max_iter=1000, random_state=SEMILLA),
    "Arbol de decision": DecisionTreeClassifier(max_depth=4, random_state=SEMILLA),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=SEMILLA),
}

pliegues = StratifiedKFold(n_splits=PLIEGUES_CV, shuffle=True, random_state=SEMILLA)
f1 = make_scorer(f1_score, zero_division=0)
f1_cv = {
    nombre: cross_val_score(armar_pipeline(m), x_ent, y_ent, cv=pliegues, scoring=f1).mean()
    for nombre, m in candidatos.items()
}
pd.Series(f1_cv, name="F1 CV").sort_values(ascending=False).round(3)

Random Forest          0.580
Arbol de decision      0.568
Regresion logistica    0.567
Dummy                  0.000
Name: F1 CV, dtype: float64

Conviene leer la diferencia entre los candidatos antes de dar por bueno al ganador: con pocos cientos de registros, una diferencia pequeña en F1 puede deberse a la partición y no al modelo.

In [11]:
elegido = max(f1_cv, key=f1_cv.get)
pipeline = armar_pipeline(candidatos[elegido]).fit(x_ent, y_ent)
print(f"Modelo elegido: {elegido}")

Modelo elegido: Random Forest


## 8. Evaluación final sobre el conjunto de prueba

Ahora sí se usa el 20 % reservado. Se reportan las métricas binarias contra `rechazo` y la matriz de confusión de 2×2 (filas: valor real; columnas: predicción).

In [12]:
prediccion = pipeline.predict(x_pru)
metricas = {
    "accuracy": accuracy_score(y_pru, prediccion),
    "precision": precision_score(y_pru, prediccion, zero_division=0),
    "recall": recall_score(y_pru, prediccion, zero_division=0),
    "f1": f1_score(y_pru, prediccion, zero_division=0),
}
print({k: round(v, 3) for k, v in metricas.items()})
pd.DataFrame(
    confusion_matrix(y_pru, prediccion),
    index=["real aprobado", "real rechazado"],
    columns=["pred. aprobado", "pred. rechazado"],
)

{'accuracy': 0.813, 'precision': 0.857, 'recall': 0.474, 'f1': 0.61}


,pred. aprobado,pred. rechazado
real aprobado,82,3
real rechazado,20,18


Para comparar, la línea base evaluada sobre el mismo conjunto:

In [13]:
base = armar_pipeline(DummyClassifier(strategy="most_frequent")).fit(x_ent, y_ent)
pred_base = base.predict(x_pru)
print(f"Linea base  accuracy: {accuracy_score(y_pru, pred_base):.3f}  "
      f"F1: {f1_score(y_pru, pred_base, zero_division=0):.3f}")

Linea base  accuracy: 0.691  F1: 0.000


La fila "real rechazado" de la matriz muestra cuántos rechazos detecta el modelo y cuántos deja pasar; ahí se lee el recall.

## 9. De probabilidad a nivel de riesgo

La función `asignar_nivel` implementa los cortes. Los bordes son explícitos: exactamente 0.33 ya es medio y exactamente 0.67 ya es alto. Una probabilidad fuera de [0, 1] no es válida y detiene la ejecución.

In [14]:
def asignar_nivel(probabilidad):
    """Convierte la probabilidad de rechazo en bajo, medio o alto."""
    if not 0 <= probabilidad <= 1:
        raise ValueError(f"Probabilidad fuera de [0, 1]: {probabilidad}")
    if probabilidad < CORTE_MEDIO:
        return NIVELES[0]
    if probabilidad < CORTE_ALTO:
        return NIVELES[1]
    return NIVELES[2]


bordes = [0.0, CORTE_MEDIO - 0.01, CORTE_MEDIO, CORTE_ALTO - 0.01, CORTE_ALTO, 1.0]
pd.DataFrame({"probabilidad": bordes, "nivel": [asignar_nivel(p) for p in bordes]})

,probabilidad,nivel
0,0.00,bajo
1,0.32,bajo
2,0.33,medio
3,0.66,medio
4,0.67,alto
5,1.00,alto


Esta tabla es el oráculo de los casos de valores límite: cada probabilidad tiene un nivel esperado sin ambigüedad.

Como no hay etiquetas reales de tres clases, los niveles se evalúan así: si son útiles, la proporción de rechazos reales debe crecer de bajo a medio a alto.

In [15]:
probabilidad = pipeline.predict_proba(x_pru)[:, 1]
niveles = pd.Series([asignar_nivel(p) for p in probabilidad], index=y_pru.index)
(
    pd.DataFrame({"nivel": niveles, "rechazo": y_pru})
    .groupby("nivel")["rechazo"].agg(registros="count", rechazo_real="mean")
    .reindex(NIVELES)
    .round(2)
)

,registros,rechazo_real
nivel,,
bajo,90,0.18
medio,15,0.40
alto,18,0.89


Al leer esta tabla hay que mirar también la columna de registros: un nivel con pocos registros produce una proporción poco estable.

## 10. Limitaciones y puntos para el diseño de pruebas

- **Supuesto de la etiqueta:** el rechazo se usa como sustituto del riesgo de morosidad; el conjunto no tiene datos de morosidad real.
- **Sin matriz de 3×3:** no existen etiquetas reales de tres clases. Es un riesgo residual a declarar en el Go/No-Go.
- **Hiperparámetros fijos:** no se hizo búsqueda; el ganador podría cambiar con otros valores.
- **Categorías desconocidas y vacíos:** el pipeline no los rechaza (los imputa o los deja en cero). La API debe validar el contrato y responder 400.
- **Hoja `Mor Riesgo Test`:** sin variable objetivo; se reserva como datos de entrada para las pruebas de la API, con oráculos de contrato y no de exactitud.
- **Oráculos disponibles:** exacto para `asignar_nivel` (tabla del paso 9); de dirección para el modelo (por ejemplo, pasar `Credit_History` de 1 a 0 no debería bajar el nivel); de consistencia (misma entrada, mismo nivel); estadístico para la calidad (umbral mínimo de F1 y orden creciente de rechazos por nivel).